# Species Distribution Models and Taxonomic Analyses of African Pipistrelloid Bats and Sokoke Scops Owl Under Climate Change Scenarios: Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR² biodiversity dataset using the `mlcroissant` library. All entities (record sets, fields, columns) are referenced by their unique `@id` values per the Croissant schema specification.

### Dataset Source
The dataset source is a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.xws9-352w/fair2.json`

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and inspect its contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint Everyday import pprint

# Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.xws9-352w/fair2.json'

# Load dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Metadata object

# Print main description
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\n")
print(f"Identifier: {metadata.identifier}\n")
print(f"License: {metadata.license}")

## 2. Data Overview
List all available record sets and their fields, printing their `@id` for use in extraction and analysis steps. This is useful for understanding what tables and columns are defined in the Croissant schema.

In [ ]:
# List record sets and fields
print("\nAvailable Record Sets and Fields (referenced by '@id'):\n")
record_sets = list(dataset.record_sets())
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  - Name: {rs.get('name', '---')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # Ensure it's always a list
    print("  Fields:")
    for fld in fields:
        if isinstance(fld, dict) and '@id' in fld:
            print(f"    - Field @id: {fld['@id']}")
        else:
            print(f"    - Field: {fld}")
    print()

## 3. Data Extraction
Extract records from a specified record set using its `@id`. You may replace the provided `record_set_id` with any other available from the schema overview above. Similarly, adjust the field selection for your analysis by using their `@id`s.

In [ ]:
# Example: Extract all data for each record set into separate DataFrames.
# You can update this code to select a specific record set if desired.

dataframes = {}
for rs in record_sets:
    record_set_id = rs['@id']
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set: {record_set_id} (rows: {len(df)})")
        print(f"Columns: {list(df.columns)}\n")

# For illustration: select one record set @id for detailed work below
# CHANGE THIS TO YOUR TARGET RECORD SET FROM THE PRINTED OUTPUT ABOVE
selected_record_set_id = list(dataframes.keys())[0] if len(dataframes) > 0 else None
if selected_record_set_id:
    print(f"First 3 rows for {selected_record_set_id}:")
    display(dataframes[selected_record_set_id].head(3))

## 4. Exploratory Data Analysis (EDA)
Demonstrate filtering and normalization using column (`@id`) references only. Replace `numeric_field_id` and `group_field_id` with the desired field `@id` from the previous section. If the record set does not have numeric columns, adjust accordingly.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ---- CONFIG: Set field @id (from previous overviews) ----
record_set_id = selected_record_set_id  # Use whichever record set you want

# Display columns for in-notebook selection
if record_set_id:
    df = dataframes[record_set_id]
    print(f"Columns for {record_set_id}:\n{list(df.columns)}\n")
    # Let's try to pick an integer or float column (manually check below)
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    else:
        numeric_field_id = None

    if numeric_field_id:
        print(f"Selected numeric field: {numeric_field_id}")
        # Example: filter and normalize
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
        display(filtered_df.head())
        # Normalizing
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' (first 3 records):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))
        # Group by another available field if possible
        possible_groups = [c for c in df.columns if c != numeric_field_id]
        group_field_id = possible_groups[0] if possible_groups else None
        if group_field_id:
            print(f"\nGrouping by field: {group_field_id}\n")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            display(grouped.head())
    else:
        print("No numeric fields found for EDA in this record set.")

## 5. Visualization
Visualize numeric field distributions or relationships. We only reference columns by field `@id`. 

You can edit the cell to select other fields as desired.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in record set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
    # Scatter with a group field if suitable
    if group_field_id and df[group_field_id].nunique() < 20:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=60)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric field found for visualization. Please inspect the columns and set 'numeric_field_id' accordingly.")

## 6. Conclusion
In this notebook, you loaded biodiversity and species distribution data curated under the FAIR² framework with `mlcroissant`. You explored record sets and fields (referenced by `@id`), loaded data, and performed basic filtering, normalization, grouping, and visualization. Adapt field and record set variables as needed for deeper dives into specific scientific questions or downstream ML tasks.

For further schema details, refer to the Croissant documentation and the dataset's schema URL.